In [1]:
import re
import pandas as pd

In [2]:
top5_2023_2024 = pd.read_csv("top5leagues_ratings_2023-2024.csv")
top5_2022_2023 = pd.read_csv("top5leagues_ratings_2022-2023.csv")
top5_2021_2022 = pd.read_csv("top5leagues_ratings_2021-2022.csv")

In [3]:
def extract_name(text):
    # Remove leading digits and spaces
    text = re.sub(r'^\d+\s*', '', text)

    first_part = text.split(',')[0]
    
    # Split the text into words while keeping hyphenated names together, including accented characters
    words = re.findall(r'\b[A-Za-zÀ-ÖØ-öø-ÿ]+(?:-[A-Za-zÀ-ÖØ-öø-ÿ]+)?\b', first_part)
    # print(words)

    player_name = []
    team_name = []

    team_started = False

    for word in words:
        uppercase_count = sum(1 for c in word if c.isupper())

        # If the word contains more than one uppercase letter and is not hyphenated, split it into player and team names
        if uppercase_count > 1 and '-' not in word and not team_started:
            # Split the word: the first part goes to the player's last name and the second part is the team name
            match = re.search(r'([a-zà-öø-ÿA-Z]+)([A-Z].*)', word)
            if match:
                player_name.append(match.group(1))  # Append the first part (last name of the player)
                team_name.append(match.group(2))    # Append the team name part
            team_started = True
        elif team_started:
            # Once team name starts, continue appending words to the team name
            team_name.append(word)
        else:
            # Add words to player name until team name starts
            player_name.append(word)

    # Check if team name is still empty and if the last part is part of the team name
    if not team_name:
        last_word = player_name[-1]
        team_match = re.search(r'(.*)(Tottenham|Man Utd|City|Chelsea|Palace|Arsenal|Liverpool|Brighton)', last_word, re.IGNORECASE)
        if team_match:
            player_name[-1] = team_match.group(1)  # Correct the player name if it's concatenated with the team
            team_name.append(team_match.group(2))  # Add the correct team name

    player = ' '.join(player_name)
    team = ' '.join(team_name)

    return player, team

In [4]:
top5_2023_2024[['Player Name', 'Squad']] = top5_2023_2024['Player Name'].apply(
    lambda x: pd.Series(extract_name(x))
)
top5_2023_2024.to_csv("cleaned_top_5_2023_2024.csv")
top5_2022_2023[['Player Name', 'Squad']] = top5_2022_2023['Player Name'].apply(
    lambda x: pd.Series(extract_name(x))
)
top5_2022_2023.to_csv("cleaned_top_5_2022_2023.csv")
top5_2021_2022[['Player Name', 'Squad']] = top5_2021_2022['Player Name'].apply(
    lambda x: pd.Series(extract_name(x))
)
top5_2021_2022.to_csv("cleaned_top_5_2021_2022.csv")

In [5]:
# !pip install rapidfuzz
from rapidfuzz import process
def get_fuzzy_rating(player, squad, df_players):
    # Get fuzzy match for player names
    best_player_match = process.extractOne(player, df_players["Player Name"].tolist(), score_cutoff=50)
    
    # Get fuzzy match for squad (club) names
    best_squad_match = process.extractOne(squad, df_players["Squad"].tolist(), score_cutoff=30)

    if best_player_match and best_squad_match:
        matched_name = best_player_match[0]
        matched_squad = best_squad_match[0]
        
        # Find the rating using both player and squad match
        matched_rating = df_players[
            (df_players["Player Name"] == matched_name) & 
            (df_players["Squad"] == matched_squad)
        ]["Rating"].values

        if matched_rating.size > 0:
            return matched_rating[0]
    return None  # Return None if no match found

In [6]:
def clean_league_name(league):
    return re.sub(r'^\S+\s+', '', league)  # Removes the first word and space

In [7]:
filtered_df = pd.read_csv("polished_data_no_ratings.csv")

In [8]:
df_2023_2024 = filtered_df[filtered_df["Season"] == "2023-2024"].copy()
df_2023_2024['League'] = df_2023_2024['League'].apply(clean_league_name)

# Merge ratings on 'Player Name' (keeping all existing players)
df_2023_2024 = pd.merge(
    df_2023_2024,
    top5_2023_2024[['Player Name', 'Squad', 'Rating']], 
    on=['Player Name', 'Squad'], 
    how='left'
)

# Apply fuzzy matching for players missing ratings
df_2023_2024["Rating"] = df_2023_2024.apply(
    lambda row: get_fuzzy_rating(row["Player Name"], row["Squad"], top5_2023_2024) 
    if pd.isnull(row["Rating"]) else row["Rating"],
    axis=1
)

In [9]:
df_2022_2023 = filtered_df[filtered_df["Season"] == "2022-2023"].copy()
df_2022_2023['League'] = df_2022_2023['League'].apply(clean_league_name)

# Merge ratings on 'Player Name' (keeping all existing players)
df_2022_2023 = pd.merge(
    df_2022_2023,
    top5_2022_2023[['Player Name', 'Squad', 'Rating']], 
    on=['Player Name', 'Squad'], 
    how='left'
)

# Apply fuzzy matching for players missing ratings
df_2022_2023["Rating"] = df_2022_2023.apply(
    lambda row: get_fuzzy_rating(row["Player Name"], row["Squad"], top5_2022_2023) 
    if pd.isnull(row["Rating"]) else row["Rating"],
    axis=1
)

In [10]:
df_2021_2022 = filtered_df[filtered_df["Season"] == "2021-2022"].copy()
df_2021_2022['League'] = df_2021_2022['League'].apply(clean_league_name)

# Merge ratings on 'Player Name' (keeping all existing players)
df_2021_2022 = pd.merge(
    df_2021_2022,
    top5_2021_2022[['Player Name', 'Squad', 'Rating']], 
    on=['Player Name', 'Squad'], 
    how='left'
)

# Apply fuzzy matching for players missing ratings
df_2021_2022["Rating"] = df_2021_2022.apply(
    lambda row: get_fuzzy_rating(row["Player Name"], row["Squad"], top5_2021_2022) 
    if pd.isnull(row["Rating"]) else row["Rating"],
    axis=1
)

In [11]:
df_2023_2024 = df_2023_2024.drop_duplicates()
df_2022_2023 = df_2022_2023.drop_duplicates()
df_2021_2022 = df_2021_2022.drop_duplicates()

In [12]:
finalized_df = pd.concat([df_2023_2024, df_2022_2023, df_2021_2022], ignore_index=True)

In [13]:
finalized_df

,Player Name,Pos,Squad,League,90s,Tkl,TklW,Def 3rd,Mid 3rd,Att 3rd,...,Off,Crs,PKcon,OG,Recov,Won,Lost,Won%,Season,Rating
0,Max Aarons,DF,Bournemouth,Premier League,13.7,29,19,20,7,2,...,2,13,1,0,75,5,11,31.3,2023-2024,6.25
1,Yunis Abdelhamid,DF,Reims,Ligue 1,30.9,64,35,36,23,5,...,0,3,0,1,149,61,37,62.2,2023-2024,6.77
2,Salis Abdul Samed,MF,Lens,Ligue 1,16.9,21,14,8,10,3,...,0,3,3,0,89,2,7,22.2,2023-2024,6.22
3,Laurent Abergel,MF,Lorient,Ligue 1,31.8,85,52,43,34,8,...,1,34,0,0,226,15,14,51.7,2023-2024,6.91
4,Abner,DF,Betis,La Liga,15.6,25,19,15,9,1,...,2,26,1,0,79,14,10,58.3,2023-2024,6.41
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3647,Nadir Zortea,"DF,MF",Salernitana,Serie A,15.7,21,15,9,7,5,...,0,85,0,0,72,8,17,32.0,2021-2022,6.36
3648,Kurt Zouma,DF,West Ham,Premier League,23.1,11,6,8,3,0,...,0,1,0,1,92,50,31,61.7,2021-2022,6.74
3649,Igor Zubeldia,DF,Real Sociedad,La Liga,18.3,20,10,12,8,0,...,1,4,1,0,90,41,34,54.7,2021-2022,6.41
3650,Martín Zubimendi,MF,Real Sociedad,La Liga,28.8,52,25,19,30,3,...,3,2,0,0,139,63,24,72.4,2021-2022,6.76


In [14]:
nan_indices = finalized_df[finalized_df.isna().any(axis=1)].index
print(nan_indices)

Index([  17,   19,   44,   50,   56,   80,   92,  126,  145,  163,
       ...
       3335, 3380, 3382, 3469, 3561, 3579, 3590, 3609, 3610, 3612],
      dtype='int64', length=131)


In [15]:
nan_count = finalized_df.isna().any(axis=1).sum()
print(nan_count)

131


In [21]:
nan_entries = finalized_df[finalized_df.isna().any(axis=1)]
nan_entries.to_excel("nan.xlsx")

In [16]:
finalized_df.to_csv("finalized_players.csv", header=True, index=False)